# Phase 1 — Class-Specialist Ensemble + Per-Class Threshold Tuning

Different hyperparameter runs win on different classes — there is no single best checkpoint.
Rather than chase one more config, this notebook combines the two strongest existing checkpoints
by routing each class to whichever model generalises best for it, then tunes a per-class confidence
threshold biased toward **recall** (this is a human-reviewed assist tool — a missed organism never
reaches the pathologist, a false flag costs them one extra glance).

**Models:** `yolo26s` (v1) · `yolo26n_v2` (v2)

**Routing decision — based on val→test generalisation gap, not a single noisy split:**

| Class | v1s val | v1s test | v1s gap | v2n val | v2n test | v2n gap | Routed to | Why |
|---|---|---|---|---|---|---|---|---|
| Trichomonas vaginalis | 0.581 | 0.510 | -0.071 | 0.631 | 0.443 | **-0.188** | **v1s** | v1s is far more stable (smaller val→test drop) and wins on test |
| Bacterial vaginosis | 0.704 | 0.684 | -0.020 | 0.741 | 0.727 | -0.014 | **v2n** | v2n wins on both splits, both stable |
| Candida spp. | 0.328 | 0.373 | +0.045 | 0.274 | 0.381 | +0.106 | **v2n** | razor-thin, noise-dominated — weak default pick, not a confident specialist case |
| Actinomyces spp. | 0.972 | 0.553 | **-0.419** | 0.915 | 0.995 | +0.080 | **v2n** | v1s catastrophically overfits (42 train instances); v2n is stable and high |

v1s nominally has a *higher val* AP for Actinomyces, but that number is misleading — it's the
overfitting the original training report flagged. We trust the val→test **gap**, not a single point.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT          = Path('../../../').resolve()
DATASET_YAML  = ROOT / 'models/organism_det/configs/dataset.yaml'
DATA_ROOT     = ROOT / 'data/organisms_data'
METRICS_DIR   = ROOT / 'models/organism_det/results/metrics'
CONFIG_DIR    = ROOT / 'models/organism_det/configs'
IMGSZ         = 1280

CLASS_NAMES = [
    'Trichomonas vaginalis',
    'Bacterial vaginosis flora shift',
    'Candida spp.',
    'Actinomyces spp.',
]

MODEL_PATHS = {
    'v1s': ROOT / 'models/organism_det/checkpoints/yolo26s/best.pt',
    'v2n': ROOT / 'models/organism_det/checkpoints/yolo26n_v2/best.pt',
}

# class_idx -> which model's detections to trust for that class
ROUTING = {0: 'v1s', 1: 'v2n', 2: 'v2n', 3: 'v2n'}

# Recall-priority policy: maximise recall subject to a precision floor that SCALES with
# each class's own baseline (F1-optimal) precision, not a blind absolute number. A fixed
# absolute floor degenerates to conf=0 ("flag everything") whenever baseline precision is
# high, since the floor sits below the entire curve. floor = max(absolute_min, relative_frac * baseline_P).
# These are policy choices (clinical priority), not measured facts -- tune freely.
PRECISION_POLICY = {
    0: dict(absolute_min=0.10, relative_frac=0.6),  # TV — most clinically critical, accept heavy over-flagging
    1: dict(absolute_min=0.20, relative_frac=0.6),  # BV — already robust, don't add reviewer burden
    2: dict(absolute_min=0.10, relative_frac=0.5),  # Candida — weak baseline, less room to demand
    3: dict(absolute_min=0.30, relative_frac=0.65), # Actinomyces — near-saturated, guard against conf=0
}

for p in MODEL_PATHS.values():
    assert p.exists(), f'Missing checkpoint: {p}'

### Load both models and capture full PR curves on val (the tuning set)

In [2]:
from ultralytics import YOLO

models = {k: YOLO(str(p)) for k, p in MODEL_PATHS.items()}

val_results = {
    k: m.val(data=str(DATASET_YAML), split='val', imgsz=IMGSZ, verbose=False)
    for k, m in models.items()
}

for k, r in val_results.items():
    assert list(r.box.ap_class_index) == [0, 1, 2, 3], f'{k}: unexpected class index order {r.box.ap_class_index}'
print('Both models validated on val split — PR curves captured.')

Ultralytics 8.4.60 🚀 Python-3.10.12 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4090, 22554MiB)
YOLO26s summary (fused): 122 layers, 9,466,728 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6893.1±1448.8 MB/s, size: 124.5 KB)
val: Scanning /home/pritamthapa/projects/cervical_cancer/ml-models/data/organisms_data/val/labels.cache... 47 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 47/47 16.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 3.8it/s 0.8s0.5s
                   all         47        143      0.698      0.586      0.646      0.376
Speed: 4.0ms preprocess, 5.9ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to /home/pritamthapa/projects/cervical_cancer/ml-models/runs/detect/val-18
Ultralytics 8.4.60 🚀 Python-3.10.12 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4090, 22554MiB)
YOLO26n summary (fused): 122 layers, 2,375,616 parameters, 0 g

### Per-class threshold tuning: baseline (max-F1) vs recall-prioritised (precision floor)

In [3]:
def f1_optimal_point(box_metric, class_idx):
    """Standard balanced operating point — argmax F1 over the confidence sweep."""
    px, p_curve, r_curve, f1_curve = box_metric.px, box_metric.p_curve[class_idx], box_metric.r_curve[class_idx], box_metric.f1_curve[class_idx]
    i = int(np.argmax(f1_curve))
    return px[i], p_curve[i], r_curve[i]


def recall_at_precision_floor(box_metric, class_idx, min_precision):
    """Lowest confidence threshold where precision still clears the floor -> max recall there."""
    px, p_curve, r_curve = box_metric.px, box_metric.p_curve[class_idx], box_metric.r_curve[class_idx]
    mask = p_curve >= min_precision
    if not mask.any():
        return f1_optimal_point(box_metric, class_idx) + ('fallback: floor unreachable, used max-F1',)
    i = np.where(mask)[0][np.argmax(r_curve[mask])]
    return px[i], p_curve[i], r_curve[i], 'recall@floor'


rows = []
thresholds = {}
precision_floors = {}
for cls_idx, model_key in ROUTING.items():
    box = val_results[model_key].box
    base_conf, base_p, base_r = f1_optimal_point(box, cls_idx)

    policy = PRECISION_POLICY[cls_idx]
    floor = max(policy['absolute_min'], policy['relative_frac'] * float(base_p))

    tuned = recall_at_precision_floor(box, cls_idx, floor)
    tuned_conf, tuned_p, tuned_r = tuned[0], tuned[1], tuned[2]
    flag = 'DEGENERATE (conf~0, flags nearly everything)' if tuned_conf < 0.02 else 'ok'

    thresholds[cls_idx] = round(float(tuned_conf), 4)
    precision_floors[cls_idx] = round(float(floor), 4)
    rows.append({
        'class': CLASS_NAMES[cls_idx],
        'routed_model': model_key,
        'baseline_conf': round(float(base_conf), 3),
        'baseline_P': round(float(base_p), 3),
        'baseline_R': round(float(base_r), 3),
        'precision_floor': round(float(floor), 3),
        'tuned_conf': round(float(tuned_conf), 3),
        'tuned_P': round(float(tuned_p), 3),
        'tuned_R': round(float(tuned_r), 3),
        'delta_R': round(float(tuned_r - base_r), 3),
        'status': flag,
    })

threshold_df = pd.DataFrame(rows)
print(threshold_df.to_string(index=False))
print(f'\nChosen per-class thresholds: {thresholds}')

degenerate = threshold_df[threshold_df.status.str.startswith('DEGENERATE')]
if not degenerate.empty:
    print(f'\n⚠️  {len(degenerate)} class(es) still hit a degenerate near-zero threshold — raise relative_frac/absolute_min for: {degenerate["class"].tolist()}')

                          class routed_model  baseline_conf  baseline_P  baseline_R  precision_floor  tuned_conf  tuned_P  tuned_R  delta_R status
          Trichomonas vaginalis          v1s          0.519       0.667       0.529            0.400       0.084    0.400    0.676    0.147     ok
Bacterial vaginosis flora shift          v2n          0.296       0.657       0.774            0.394       0.023    0.395    0.857    0.083     ok
                   Candida spp.          v2n          0.099       0.442       0.421            0.221       0.035    0.232    0.421    0.000     ok
               Actinomyces spp.          v2n          0.777       1.000       0.833            0.650       0.133    0.650    0.833    0.000     ok

Chosen per-class thresholds: {0: 0.0841, 1: 0.023, 2: 0.035, 3: 0.1331}


### Build the ensemble predictor (class-routed, threshold-filtered)

In [4]:
def predict_ensemble(img_path, models, routing, thresholds, imgsz=IMGSZ):
    """Run both models, keep each class's detections only from its routed model, filter by its tuned threshold."""
    raw = {k: m.predict(str(img_path), imgsz=imgsz, conf=0.001, verbose=False)[0] for k, m in models.items()}
    detections = []
    for cls_idx, model_key in routing.items():
        thr = thresholds[cls_idx]
        for box in raw[model_key].boxes:
            c = int(box.cls)
            if c != cls_idx:
                continue
            conf = float(box.conf)
            if conf < thr:
                continue
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            detections.append((c, conf, x1, y1, x2, y2))
    return detections

### Confirm once on the test set (held out from all tuning decisions above)

In [5]:
from PIL import Image

def load_yolo_labels(label_path, img_w, img_h):
    boxes = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text().strip().splitlines():
        if not line.strip():
            continue
        cls, cx, cy, w, h = map(float, line.split())
        cls = int(cls)
        x1, y1 = (cx - w / 2) * img_w, (cy - h / 2) * img_h
        x2, y2 = (cx + w / 2) * img_w, (cy + h / 2) * img_h
        boxes.append((cls, x1, y1, x2, y2))
    return boxes


def iou(a, b):
    xA, yA = max(a[0], b[0]), max(a[1], b[1])
    xB, yB = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (a[2] - a[0]) * (a[3] - a[1])
    areaB = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (areaA + areaB - inter + 1e-9)


test_images = sorted((DATA_ROOT / 'test/images').glob('*.jpg'))
tp = {c: 0 for c in range(4)}
fp = {c: 0 for c in range(4)}
fn = {c: 0 for c in range(4)}

for img_path in test_images:
    img = Image.open(img_path)
    w, h = img.size
    label_path = DATA_ROOT / 'test/labels' / f'{img_path.stem}.txt'
    gt_boxes = load_yolo_labels(label_path, w, h)
    preds = predict_ensemble(img_path, models, ROUTING, thresholds)

    for cls_idx in range(4):
        gt_c = [b[1:] for b in gt_boxes if b[0] == cls_idx]
        pred_c = sorted([p for p in preds if p[0] == cls_idx], key=lambda x: -x[1])
        matched = [False] * len(gt_c)
        for _, conf, x1, y1, x2, y2 in pred_c:
            best_iou, best_j = 0.0, -1
            for j, gtb in enumerate(gt_c):
                if matched[j]:
                    continue
                v = iou((x1, y1, x2, y2), gtb)
                if v > best_iou:
                    best_iou, best_j = v, j
            if best_iou >= 0.5:
                matched[best_j] = True
                tp[cls_idx] += 1
            else:
                fp[cls_idx] += 1
        fn[cls_idx] += matched.count(False)

final_rows = []
for cls_idx in range(4):
    P = tp[cls_idx] / (tp[cls_idx] + fp[cls_idx]) if (tp[cls_idx] + fp[cls_idx]) else 0.0
    R = tp[cls_idx] / (tp[cls_idx] + fn[cls_idx]) if (tp[cls_idx] + fn[cls_idx]) else 0.0
    final_rows.append({
        'class': CLASS_NAMES[cls_idx],
        'routed_model': ROUTING[cls_idx],
        'conf_threshold': thresholds[cls_idx],
        'TP': tp[cls_idx], 'FP': fp[cls_idx], 'FN': fn[cls_idx],
        'precision': round(P, 4), 'recall': round(R, 4),
    })

ensemble_test_df = pd.DataFrame(final_rows)
print(ensemble_test_df.to_string(index=False))

out_path = METRICS_DIR / 'ensemble_test_eval.csv'
ensemble_test_df.to_csv(out_path, index=False)
print(f'\nSaved -> {out_path}')

                          class routed_model  conf_threshold  TP  FP  FN  precision  recall
          Trichomonas vaginalis          v1s          0.0841  26  31  19     0.4561  0.5778
Bacterial vaginosis flora shift          v2n          0.0230  56 122  12     0.3146  0.8235
                   Candida spp.          v2n          0.0350  20  53  16     0.2740  0.5556
               Actinomyces spp.          v2n          0.1331   6   2   0     0.7500  1.0000

Saved -> /home/pritamthapa/projects/cervical_cancer/ml-models/models/organism_det/results/metrics/ensemble_test_eval.csv


### Save the production artifact — model routing + thresholds

In [6]:
import yaml

ensemble_config = {
    'description': 'Class-specialist ensemble: each class routed to whichever model generalises best for it (val->test gap), thresholds tuned for recall at a per-class precision floor.',
    'models': {k: str(p.relative_to(ROOT)) for k, p in MODEL_PATHS.items()},
    'imgsz': IMGSZ,
    'classes': [
        {
            'id': cls_idx,
            'name': CLASS_NAMES[cls_idx],
            'routed_model': ROUTING[cls_idx],
            'min_precision_floor': precision_floors[cls_idx],
            'conf_threshold': thresholds[cls_idx],
        }
        for cls_idx in range(4)
    ],
}

out_path = CONFIG_DIR / 'ensemble_config.yaml'
with open(out_path, 'w') as f:
    yaml.safe_dump(ensemble_config, f, sort_keys=False)
print(f'Saved -> {out_path}\n')
print(yaml.safe_dump(ensemble_config, sort_keys=False))

Saved -> /home/pritamthapa/projects/cervical_cancer/ml-models/models/organism_det/configs/ensemble_config.yaml

description: 'Class-specialist ensemble: each class routed to whichever model generalises
  best for it (val->test gap), thresholds tuned for recall at a per-class precision
  floor.'
models:
  v1s: models/organism_det/checkpoints/yolo26s/best.pt
  v2n: models/organism_det/checkpoints/yolo26n_v2/best.pt
imgsz: 1280
classes:
- id: 0
  name: Trichomonas vaginalis
  routed_model: v1s
  min_precision_floor: 0.4
  conf_threshold: 0.0841
- id: 1
  name: Bacterial vaginosis flora shift
  routed_model: v2n
  min_precision_floor: 0.3939
  conf_threshold: 0.023
- id: 2
  name: Candida spp.
  routed_model: v2n
  min_precision_floor: 0.2209
  conf_threshold: 0.035
- id: 3
  name: Actinomyces spp.
  routed_model: v2n
  min_precision_floor: 0.6499
  conf_threshold: 0.1331

